# HRFLM (Hybrid Random Forest + Linear Model) -- Global Explainability

HRFLM provides **global** explanation of overall model behavior (audience: developers/researchers/authorized model analysts -- in this application, restricted to the **admin** role only, unlike LIME which is patient/doctor-facing).

Where LIME (`lime_eval.ipynb`) explains ONE prediction at a time by fitting a local linear surrogate in the neighborhood of a single patient, HRFLM explains the model's behavior ACROSS THE ENTIRE reference population by fitting a hybrid Random Forest + Linear Model surrogate trained to approximate the real PPO model's decisions everywhere, then reporting a single global feature-importance ranking derived from both sub-models.

This notebook:
1. Loads the real, trained PPO model and applies the exact same preprocessing as production (`backend/services/preprocessingService.js`) and as `lime_eval.ipynb`.
2. Trains the hybrid RF + Linear Model surrogate against the real model's own predictions.
3. Produces a combined global feature-importance report.
4. Exports `models/hrflm/<version>/hrflm_report.json` + `metadata.json` for the backend's admin-only global-explanation view (`backend/services/hrflmService.js`).

### 1. Import Libraries

In [ ]:
import os
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

### 2. Device

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running Device :", DEVICE)

### 3. Canonical Feature Contract

In [ ]:
CANONICAL_FEATURE_ORDER = [
    "thalach", "restecg", "oldpeak", "slope", "age",
    "sex", "cp", "exang", "trestbps", "fbs"
]

FEATURE_LABELS = {
    "thalach": "Maximum heart rate",
    "restecg": "Resting ECG result",
    "oldpeak": "ECG stress-test change",
    "slope": "ECG ST-segment slope",
    "age": "Age",
    "sex": "Sex",
    "cp": "Chest pain type",
    "exang": "Exercise-related chest discomfort",
    "trestbps": "Resting blood pressure",
    "fbs": "Fasting blood sugar indicator",
}

### 4. Preprocessing Parameters (copied from `models/rl/preprocessing_params.json`)

Identical to the values used in `lime_eval.ipynb` -- see that notebook for the verification that these reproduce the original training data exactly.

In [ ]:
ROBUST_SCALER_PARAMS = {
    "age":      {"median": 56.0,  "iqr": 12.0},
    "trestbps": {"median": 130.0, "iqr": 22.0},
    "thalach":  {"median": 140.0, "iqr": 38.5},
    "oldpeak":  {"median": 1.0,   "iqr": 1.9},
}

CP_INT_TO_ONEHOT = {
    0: "cp_typical angina",
    1: "cp_atypical angina",
    2: "cp_non-anginal",
    3: "cp_asymptomatic",
}
RESTECG_INT_TO_ONEHOT = {
    0: "restecg_normal",
    1: "restecg_st-t abnormality",
    2: "restecg_lv hypertrophy",
}

TRAINED_FEATURE_ORDER_15DIM = [
    "sex", "fbs", "exang", "age", "trestbps", "thalach", "oldpeak",
    "cp_asymptomatic", "cp_atypical angina", "cp_non-anginal", "cp_typical angina",
    "restecg_lv hypertrophy", "restecg_normal", "restecg_st-t abnormality",
    "slope",
]


def to_trained_representation(features: dict) -> np.ndarray:
    """Raw 10 canonical features -> the 15-dim vector the RL models were
    trained on. Direct port of backend/services/preprocessingService.js."""
    vector = {}
    vector["sex"] = features["sex"]
    vector["fbs"] = features["fbs"]
    vector["exang"] = features["exang"]
    vector["slope"] = features["slope"]

    for key in ("age", "trestbps", "thalach", "oldpeak"):
        params = ROBUST_SCALER_PARAMS[key]
        vector[key] = (features[key] - params["median"]) / params["iqr"]

    for column in ["cp_asymptomatic", "cp_atypical angina", "cp_non-anginal", "cp_typical angina",
                   "restecg_lv hypertrophy", "restecg_normal", "restecg_st-t abnormality"]:
        vector[column] = 0
    vector[CP_INT_TO_ONEHOT[features["cp"]]] = 1
    vector[RESTECG_INT_TO_ONEHOT[features["restecg"]]] = 1

    return np.array([vector[key] for key in TRAINED_FEATURE_ORDER_15DIM], dtype=np.float32)

### 5. Load the Trained PPO Model

In [ ]:
HIDDEN_DIM = 128
ACTION_SIZE = 2
STATE_SIZE = 15

PPO_MODEL_PATH = "../../trained_models/ppo.pth"

if not os.path.exists(PPO_MODEL_PATH):
    raise FileNotFoundError(f"{PPO_MODEL_PATH} not found. Update PPO_MODEL_PATH.")


class ActorCritic(nn.Module):
    def __init__(self, state_size):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_size, HIDDEN_DIM), nn.ReLU(),
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM), nn.ReLU(),
        )
        self.actor = nn.Sequential(
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM), nn.ReLU(),
            nn.Linear(HIDDEN_DIM, ACTION_SIZE), nn.Softmax(dim=-1),
        )
        self.critic = nn.Sequential(
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM), nn.ReLU(),
            nn.Linear(HIDDEN_DIM, 1),
        )
        self.to(DEVICE)

    def forward(self, state):
        if not torch.is_tensor(state):
            state = torch.FloatTensor(state)
        state = state.to(DEVICE)
        features = self.shared(state)
        return self.actor(features), self.critic(features)


ppo_model = ActorCritic(STATE_SIZE)
checkpoint = torch.load(PPO_MODEL_PATH, map_location=DEVICE)
ppo_model.load_state_dict(checkpoint["actor_state_dict"])
ppo_model.critic.load_state_dict(checkpoint["critic_state_dict"])
ppo_model.eval()

print("PPO model loaded from:", PPO_MODEL_PATH)

MODEL_VERSION_EXPLAINED = "ppo-fold4-2026-08-13"  # matches models/rl/ppo-fold4-2026-08-13/metadata.json

### 6. Prediction Function

In [ ]:
def predict_probability_from_raw(raw_feature_rows: np.ndarray) -> np.ndarray:
    """raw_feature_rows: shape (n, 10), CANONICAL_FEATURE_ORDER, raw values."""
    preprocessed_rows = []
    for row in raw_feature_rows:
        features = dict(zip(CANONICAL_FEATURE_ORDER, row))
        for key in ("restecg", "slope", "sex", "cp", "exang", "fbs"):
            features[key] = int(round(features[key]))
        preprocessed_rows.append(to_trained_representation(features))

    batch = torch.FloatTensor(np.stack(preprocessed_rows)).to(DEVICE)
    with torch.no_grad():
        action_probabilities, _ = ppo_model(batch)
    return action_probabilities.cpu().numpy()

### 7. Load Reference Data and Get the Real Model's Predictions

The hybrid surrogate is trained to approximate the REAL PPO model's own predicted class for every patient in the cleaned reference dataset -- not the ground-truth diagnosis label. This is what makes it a faithful **explanation of the model**, rather than an independently-trained second classifier (consistent with the same requirement applied to LIME, applied here to HRFLM as well).

In [ ]:
REFERENCE_DATA_PATH = "../data/heart_disease_trimmed.csv"

reference_df = pd.read_csv(REFERENCE_DATA_PATH)
X_raw = reference_df[CANONICAL_FEATURE_ORDER].to_numpy(dtype=np.float32)
y_true = reference_df["target"].to_numpy(dtype=np.int64)

real_model_probabilities = predict_probability_from_raw(X_raw)
y_model = np.argmax(real_model_probabilities, axis=1)  # surrogate TRAINING target

print("Reference data shape:", X_raw.shape)
print("Real model predicted-positive rate:", y_model.mean().round(4))
print("Ground-truth positive rate         :", y_true.mean().round(4))

### 8. Train the Hybrid Surrogate (Random Forest + Linear Model)

In [ ]:
RANDOM_SEED = 42

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

random_forest = RandomForestClassifier(
    n_estimators=200, max_depth=6, random_state=RANDOM_SEED, class_weight="balanced"
)
random_forest.fit(X_raw, y_model)

linear_model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_SEED)
linear_model.fit(X_scaled, y_model)

print("Random Forest trained. OOB-style training accuracy:", round(random_forest.score(X_raw, y_model), 4))
print("Logistic Regression trained. Training accuracy    :", round(linear_model.score(X_scaled, y_model), 4))

### 9. Hybrid Prediction Function

Averages the Random Forest's and the Logistic Regression's class-1 probability to produce one combined "hybrid" surrogate prediction -- this IS the HRFLM surrogate used for all metrics below and in `hrflm_eval.ipynb`.

In [ ]:
def hrflm_predict_probability(raw_rows: np.ndarray) -> np.ndarray:
    """Hybrid RF + Linear Model probability, shape (n, 2)."""
    rf_probabilities = random_forest.predict_proba(raw_rows)
    lr_probabilities = linear_model.predict_proba(scaler.transform(raw_rows))
    hybrid_probabilities = (rf_probabilities + lr_probabilities) / 2.0
    return hybrid_probabilities


hybrid_probabilities_all = hrflm_predict_probability(X_raw)
hybrid_predictions_all = np.argmax(hybrid_probabilities_all, axis=1)

print("Hybrid surrogate accuracy vs. real model predictions:",
      round(float((hybrid_predictions_all == y_model).mean()), 4))

### 10. Global Feature Importance (the HRFLM explanation)

Combines Random Forest Gini importances with |Logistic Regression coefficients|, each normalized to sum to 1, then averaged -- giving one ranked list of which raw features most drive the PPO model's decisions ACROSS THE WHOLE POPULATION (a global explanation), in contrast to LIME's per-patient local contribution scores.

In [ ]:
rf_importance = random_forest.feature_importances_
rf_importance_normalized = rf_importance / rf_importance.sum() if rf_importance.sum() > 0 else rf_importance

lr_importance = np.abs(linear_model.coef_[0])
lr_importance_normalized = lr_importance / lr_importance.sum() if lr_importance.sum() > 0 else lr_importance

hybrid_importance = (rf_importance_normalized + lr_importance_normalized) / 2.0

global_feature_importance = sorted(
    [
        {
            "feature": feature,
            "label": FEATURE_LABELS[feature],
            "random_forest_importance": float(rf_importance_normalized[i]),
            "linear_model_importance": float(lr_importance_normalized[i]),
            "hybrid_importance": float(hybrid_importance[i]),
        }
        for i, feature in enumerate(CANONICAL_FEATURE_ORDER)
    ],
    key=lambda entry: entry["hybrid_importance"],
    reverse=True,
)

print("=" * 70)
print("GLOBAL FEATURE IMPORTANCE (HRFLM)")
print("=" * 70)
for entry in global_feature_importance:
    print(f"{entry['label']:40s} {entry['hybrid_importance']:.4f}  "
          f"(RF={entry['random_forest_importance']:.4f}, LM={entry['linear_model_importance']:.4f})")
print("=" * 70)

### 11. Export HRFLM Report and Metadata

Exports to `models/hrflm/<version>/` -- consumed by `backend/services/hrflmService.js` for the admin-only global-explanation view. Also saves the fitted scaler/RF/LR objects so this exact surrogate (not a re-trained one) can be reloaded for further offline analysis if needed.

In [ ]:
import pickle

HRFLM_VERSION = f"hrflm-{datetime.now(timezone.utc).strftime('%Y-%m-%d')}"
OUTPUT_DIR = os.path.join("..", "..", "models", "hrflm", HRFLM_VERSION)
os.makedirs(OUTPUT_DIR, exist_ok=True)

hrflm_report = {
    "hrflm_version": HRFLM_VERSION,
    "explains_model_version": MODEL_VERSION_EXPLAINED,
    "reference_dataset": "research/data/heart_disease_trimmed.csv",
    "reference_row_count": int(len(reference_df)),
    "global_feature_importance": global_feature_importance,
    "surrogate_training_metadata": {
        "surrogate_target": "real_model_predicted_class",
        "random_forest": {"n_estimators": 200, "max_depth": 6, "random_state": RANDOM_SEED},
        "linear_model": {"type": "LogisticRegression", "max_iter": 1000, "random_state": RANDOM_SEED},
        "hybrid_combination": "average of RF and LM class-1 probability",
    },
    "surrogate_fit_quality": {
        "hybrid_accuracy_vs_real_model": float((hybrid_predictions_all == y_model).mean()),
        "random_forest_accuracy_vs_real_model": float(random_forest.score(X_raw, y_model)),
        "linear_model_accuracy_vs_real_model": float(linear_model.score(X_scaled, y_model)),
    },
    "generated_at": datetime.now(timezone.utc).isoformat(),
}

report_path = os.path.join(OUTPUT_DIR, "hrflm_report.json")
with open(report_path, "w") as f:
    json.dump(hrflm_report, f, indent=2)
print(f"Saved: {report_path}")

metadata = {
    "model_version": HRFLM_VERSION,
    "explains_model_version": MODEL_VERSION_EXPLAINED,
    "algorithm": "HRFLM",
    "feature_order": CANONICAL_FEATURE_ORDER,
    "audience": "admin-only",
    "training_metadata": {
        "reference_dataset": "research/data/heart_disease_trimmed.csv",
        "trained_on": datetime.now(timezone.utc).strftime("%Y-%m-%d"),
    },
    "runtime_compatibility": {
        "export_format": "json",
        "report_file": "hrflm_report.json",
    },
}
metadata_path = os.path.join(OUTPUT_DIR, "metadata.json")
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)
print(f"Saved: {metadata_path}")

with open(os.path.join(OUTPUT_DIR, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)
with open(os.path.join(OUTPUT_DIR, "random_forest.pkl"), "wb") as f:
    pickle.dump(random_forest, f)
with open(os.path.join(OUTPUT_DIR, "linear_model.pkl"), "wb") as f:
    pickle.dump(linear_model, f)
print("Saved fitted scaler.pkl, random_forest.pkl, linear_model.pkl (for reproducibility/offline analysis).")

try:
    from google.colab import files
    files.download(report_path)
    files.download(metadata_path)
    print("\nDownload triggered for hrflm_report.json and metadata.json")
except ImportError:
    print(f"\nNot running in Google Colab -- files are available locally under: {OUTPUT_DIR}")